# Notebook that focuses on a ertain rectangle in Spain. FOCUS SUD

---

In [ ]:
# global imports 
import sys
import os
import s3fs
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# local imports 
from matplotlib.patches import Rectangle
from scipy.spatial import cKDTree
from tqdm import tqdm#########




In [ ]:
MY_BUCKET = "matheo"
CHEMIN_FICHIER = "diffusion/data"

FICHIER = "data_all_years_spain.parquet"

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

with fs.open(f"s3://{MY_BUCKET}/{CHEMIN_FICHIER}/{FICHIER}") as f:
    df = pd.read_parquet(f)

In [ ]:
# decomment to make the code faster
#df = df[df['time'].isin( pd.date_range("1996-01-01", "1996-12-31", freq="D").strftime("%Y-%m-%d").tolist())]
#df2 = df.copy()

df = df.dropna(subset=['fwi-daily-proj'])

In [ ]:
from matplotlib.patches import Rectangle

GRID_SIZE = 28
VALUE_COL = "fwi-daily-proj"

# =========================================================
# 1. ZOOM 28x28 SUR LE SUD-EST DE L'ESPAGNE
# =========================================================
# Fenêtre sud-est : Andalousie orientale / Murcie / Valence
SE_LON_MIN, SE_LON_MAX = -8.7, -0.2
SE_LAT_MIN, SE_LAT_MAX = 37.2, 41.0

se_lon_edges   = np.linspace(SE_LON_MIN, SE_LON_MAX, GRID_SIZE + 1)
se_lat_edges   = np.linspace(SE_LAT_MIN, SE_LAT_MAX, GRID_SIZE + 1)
se_lon_centers = 0.5 * (se_lon_edges[:-1] + se_lon_edges[1:])
se_lat_centers = 0.5 * (se_lat_edges[:-1] + se_lat_edges[1:])

def image_aggregate_box(sub_df, lon_edges_, lat_edges_, how="mean", print_=True):
    """ """

    ix = np.digitize(sub_df["lon"].values, lon_edges_) - 1
    iy = np.digitize(sub_df["lat"].values, lat_edges_) - 1
    mask = (ix >= 0) & (ix < GRID_SIZE) & (iy >= 0) & (iy < GRID_SIZE)
    ix, iy = ix[mask], iy[mask]
    vals = sub_df[VALUE_COL].values[mask]

    if print_ == True:
        print("*** Shape number of values ***")
        print(vals.shape)

    img = np.full((GRID_SIZE, GRID_SIZE), np.nan)
    if how == "mean":
        sums   = np.zeros((GRID_SIZE, GRID_SIZE))
        counts = np.zeros((GRID_SIZE, GRID_SIZE))
        np.add.at(sums,   (iy, ix), vals)
        np.add.at(counts, (iy, ix), 1)
        with np.errstate(invalid="ignore"):
            img = np.where(counts > 0, sums / counts, np.nan)
    return img


df_se = df[(df["lon"] >= SE_LON_MIN) & (df["lon"] <= SE_LON_MAX) &
           (df["lat"] >= SE_LAT_MIN) & (df["lat"] <= SE_LAT_MAX)]

images_se_mean    = {}
all_dates = np.sort(df["time"].unique())

i = -1
for d in tqdm(all_dates, desc="Zoom sud-est"):
    i+=1
    sub = df_se[df_se["time"] == d]
    if len(sub) == 0:
        continue
    if i == 0:
        images_se_mean[d] = image_aggregate_box(sub, se_lon_edges, se_lat_edges, "mean", print_=True)
    else:
        images_se_mean[d] = image_aggregate_box(sub, se_lon_edges, se_lat_edges, "mean", print_=False)
    
print(f"{len(images_se_mean)} images 28x28 zoom sud-est construites.")

# =========================================================
# 2. PLOT : grille principale + rectangle rouge indiquant le zoom
# =========================================================
se_extent = [SE_LON_MIN, SE_LON_MAX, SE_LAT_MIN, SE_LAT_MAX]

dates_plot = ['1996-01-01','1996-02-01','1996-03-01',
              '1996-04-01','1996-05-01','1996-06-01',
              '1996-07-01','1996-08-01','1996-09-01',
              '1996-10-01','1996-11-01','1996-12-01']
dates_plot = [np.datetime64(d) for d in dates_plot]

fig, ax = plt.subplots(figsize=(10, 8))
temp = df[df["time"] == dates_plot[0]]
sc = ax.scatter(temp["lon"], temp["lat"], c=temp[VALUE_COL],
                s=2, cmap="viridis")
rect = Rectangle((SE_LON_MIN, SE_LAT_MIN),
                 SE_LON_MAX - SE_LON_MIN,
                 SE_LAT_MAX - SE_LAT_MIN,
                 linewidth=2, edgecolor="red", facecolor="none",
                 label="Fenêtre")
ax.add_patch(rect)
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.set_title(f"Position de la fen^tre 28x28 sur l'Espagne")
ax.legend()
plt.colorbar(sc, ax=ax, label=VALUE_COL)
plt.tight_layout()
plt.show()


def plot_zoom(images_dict, title):
    stack = np.stack([images_dict[d] for d in dates_plot if d in images_dict])
    vmin, vmax = np.nanmin(stack), np.nanmax(stack)

    fig, axes = plt.subplots(3, 4, figsize=(18, 12))
    axes = axes.flatten()
    for i, d in enumerate(dates_plot):
        if d not in images_dict:
            axes[i].axis("off")
            continue
        im = axes[i].imshow(images_dict[d], origin="lower", extent=se_extent,
                            vmin=vmin, vmax=vmax, cmap="viridis")
        axes[i].set_title(str(d)[:10])
        axes[i].set_xlabel("lon"); axes[i].set_ylabel("lat")
        plt.colorbar(im, ax=axes[i], fraction=0.04)
    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


# --- Zoom 28x28 sur le sud-est ---
plot_zoom(images_se_mean,    "Zoom Sud-Est 28x28 — moyenne")

In [ ]:

# =========================================================
# 3. REMPLACEMENT DES NaN PAR UNE VALEUR NÉGATIVE FIXE
# =========================================================

# Valeur de remplissage : - max de la valeur positive sur l'ensemble du dataset
global_max = df[VALUE_COL].max()
fill_value = 0.0
print(f"Valeur max globale : {global_max:.4f}")
print(f"Valeur de remplissage NaN : {fill_value:.4f}")

# Remplissage des NaN dans chaque image 28x28
images_se_mean_filled = {
    d: np.where(np.isnan(img), fill_value, img)
    for d, img in images_se_mean.items()
}

# Vérification rapide
n_nan_before = sum(np.isnan(img).sum() for img in images_se_mean.values())
n_nan_after  = sum(np.isnan(img).sum() for img in images_se_mean_filled.values())
print(f"NaN avant remplissage : {n_nan_before}")
print(f"NaN après remplissage : {n_nan_after}")

# Replot rapide pour vérifier visuellement que les NaN sont bien comblés
plot_zoom(images_se_mean_filled, f"Zoom Sud-Est 28x28 — moyenne (NaN → {fill_value:.2f})")


# =========================================================
# 4. CONSTRUCTION DU DATAFRAME LONG FORMAT (time, lat, lon, value)
# =========================================================
# Chaque pixel (iy, ix) correspond à (se_lat_centers[iy], se_lon_centers[ix])
LON_GRID, LAT_GRID = np.meshgrid(se_lon_centers, se_lat_centers)  # shape (28, 28)
lon_flat = LON_GRID.ravel()
lat_flat = LAT_GRID.ravel()
n_pixels = GRID_SIZE * GRID_SIZE  # 784

records = []
for d, img in tqdm(images_se_mean_filled.items(), desc="Construction parquet"):
    records.append(pd.DataFrame({
        "time": np.repeat(np.datetime64(d), n_pixels),
        "lat":  lat_flat,
        "lon":  lon_flat,
        VALUE_COL: img.ravel(),
    }))

df_out = pd.concat(records, ignore_index=True)
print(f"DataFrame final : {df_out.shape}")
print(df_out.head())

---

In [ ]:
# =========================================================
# 5. SAUVEGARDE EN PARQUET SUR MINIO (S3)
# =========================================================
CHEMIN_FICHIER_OUT = "diffusion/data/fwi_se_spain_28x28.parquet"

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

with fs.open(f"s3://{MY_BUCKET}/{CHEMIN_FICHIER_OUT}", "wb") as f:
    df_out.to_parquet(f, index=False)

print(f"✓ Enregistré sur S3 : s3://{MY_BUCKET}/{CHEMIN_FICHIER_OUT}")
print(f"  {df_out.shape[0]} lignes x {df_out.shape[1]} colonnes")

---

In [ ]:
GRID_SIZE = 28
N_NEIGHBORS = GRID_SIZE * GRID_SIZE  # 784
VALUE_COL = "fwi-daily-proj"

# =========================================================
# 1. POINT DE RÉFÉRENCE (à ajuster comme tu veux)
# =========================================================

# Exemple : centre sur Murcie, sud-est de l'Espagne
REF_LON, REF_LAT = -3, 39.5

# =========================================================
# 2. IDENTIFICATION DES 784 POINTS LES PLUS PROCHES (1 seule fois)
# =========================================================
# On suppose que la grille spatiale est la même pour toutes les dates,
# donc on prend les coords d'une date quelconque pour définir les voisins.
first_date = df["time"].iloc[0]
coords = df[df["time"] == first_date][["lon", "lat"]].values

tree = cKDTree(coords)
_, neighbor_idx = tree.query([REF_LON, REF_LAT], k=N_NEIGHBORS)
neighbor_coords = coords[neighbor_idx]  # shape (784, 2)

print(f"Voisinage sélectionné : {len(neighbor_coords)} points")
print(f"Emprise : lon ∈ [{neighbor_coords[:,0].min():.2f}, {neighbor_coords[:,0].max():.2f}], "
      f"lat ∈ [{neighbor_coords[:,1].min():.2f}, {neighbor_coords[:,1].max():.2f}]")

# =========================================================
# 3. RANGEMENT DES 784 POINTS DANS UNE MATRICE 28x28
# =========================================================
# On trie d'abord par latitude décroissante (haut = nord), puis par longitude
# dans chaque rangée. Ça fonctionne proprement si les points forment une grille.
order = np.lexsort((neighbor_coords[:, 0], -neighbor_coords[:, 1]))
ordered_coords = neighbor_coords[order]
ordered_idx_in_coords = neighbor_idx[order]  # index des points dans `coords`

# Vérification : la forme est-elle vraiment un carré régulier ?
lats_sorted_desc = np.sort(np.unique(np.round(ordered_coords[:, 1], 6)))[::-1]
lons_sorted_asc  = np.sort(np.unique(np.round(ordered_coords[:, 0], 6)))
print(f"Nb de latitudes uniques : {len(lats_sorted_desc)} | "
      f"Nb de longitudes uniques : {len(lons_sorted_asc)}")
if len(lats_sorted_desc) != GRID_SIZE or len(lons_sorted_asc) != GRID_SIZE:
    print("⚠ Le voisinage n'est pas exactement un carré 28x28 régulier.")
    print("  → le tri (lat décroissante, lon croissante) reste valable mais")
    print("    les rangées/colonnes peuvent être un peu irrégulières.")

# Bounding box du voisinage (utile pour le rectangle rouge)
BOX_LON_MIN = neighbor_coords[:, 0].min()
BOX_LON_MAX = neighbor_coords[:, 0].max()
BOX_LAT_MIN = neighbor_coords[:, 1].min()
BOX_LAT_MAX = neighbor_coords[:, 1].max()

# =========================================================
# 4. CONSTRUCTION DES IMAGES 28x28 POUR TOUTES LES DATES
# =========================================================
# On utilise les coordonnées des voisins comme clé : pour chaque date, on
# récupère la valeur de chaque voisin via un merge.
neighbor_df = pd.DataFrame({
    "lon": ordered_coords[:, 0],
    "lat": ordered_coords[:, 1],
    "pixel_order": np.arange(N_NEIGHBORS),  # 0..783, ordre de rangement
})

all_dates = np.sort(df["time"].unique())
images_bfs = {}

# On restreint d'abord df aux seules coords des voisins pour accélérer
df_neigh = df.merge(neighbor_df, on=["lon", "lat"], how="inner")

for d, sub in tqdm(df_neigh.groupby("time"), desc="Images 28x28 voisinage",
                   total=df_neigh["time"].nunique()):
    if len(sub) != N_NEIGHBORS:
        # par sécurité, on saute les dates incomplètes
        continue
    sub_sorted = sub.sort_values("pixel_order")
    img = sub_sorted[VALUE_COL].values.reshape(GRID_SIZE, GRID_SIZE)
    images_bfs[d] = img

print(f"{len(images_bfs)} images 28x28 voisinage construites.")

# =========================================================
# 5. PLOT : Espagne complète avec point + rectangle rouge du voisinage
# =========================================================
dates_plot = ['1996-01-01','1996-02-01','1996-03-01',
              '1996-04-01','1996-05-01','1996-06-01',
              '1996-07-01','1996-08-01','1996-09-01',
              '1996-10-01','1996-11-01','1996-12-01']
dates_plot = [np.datetime64(d) for d in dates_plot]

# Échelle de couleur commune sur les 12 mois affichés
stack = np.stack([images_bfs[d] for d in dates_plot if d in images_bfs])
vmin, vmax = np.nanmin(stack), np.nanmax(stack)

# --- Carte Espagne globale avec localisation du voisinage ---
fig, ax = plt.subplots(figsize=(10, 8))
temp = df[df["time"] == dates_plot[0]]
sc = ax.scatter(temp["lon"], temp["lat"], c=temp[VALUE_COL],
                s=2, cmap="viridis")
ax.scatter([REF_LON], [REF_LAT], s=120, facecolor="red",
           edgecolor="black", zorder=5, label="Point de référence")
rect = Rectangle((BOX_LON_MIN, BOX_LAT_MIN),
                 BOX_LON_MAX - BOX_LON_MIN,
                 BOX_LAT_MAX - BOX_LAT_MIN,
                 linewidth=2, edgecolor="red", facecolor="none",
                 label="Voisinage 28x28")
ax.add_patch(rect)
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.set_title(f"Position du voisinage 28x28 autour de ({REF_LON}, {REF_LAT})")
ax.legend()
plt.colorbar(sc, ax=ax, label=VALUE_COL)
plt.tight_layout()
plt.show()

# --- Les 12 images 28x28 voisinage ---
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, d in enumerate(dates_plot):
    if d not in images_bfs:
        axes[i].axis("off")
        continue
    im = axes[i].imshow(images_bfs[d], origin="upper",
                        extent=[BOX_LON_MIN, BOX_LON_MAX, BOX_LAT_MIN, BOX_LAT_MAX],
                        vmin=vmin, vmax=vmax, cmap="viridis")
    axes[i].set_title(str(d)[:10])
    axes[i].set_xlabel("lon"); axes[i].set_ylabel("lat")
    plt.colorbar(im, ax=axes[i], fraction=0.04)
fig.suptitle(f"Image 28x28 — 784 plus proches voisins de ({REF_LON}, {REF_LAT})",
             fontsize=16)
plt.tight_layout()
plt.show()